# User-Based Collaborative Filtering (UBCF)

 User-Based CF focuses on similarity between users on a specific item. 

### Step 1: Import all necessary libraries

Import all important library before running the notebook

In [12]:
import pandas as pd
import numpy as np
import time
from math import sqrt
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity

### Step 2: Load the dataset

u.data gives the actual user-movie-rating triples, while u.item gives the movies metadata (titles, genres, etc) that is important to display human-readable recommendations instead of just providing movie IDs.

In [13]:
# Import data of users
user_df = pd.read_csv('C:/Lecture Notes/Degree/Sem 9/TDA - Algorithm and Analysis/Collaborative-Filtering-Algorithms-in-Recommendation-Systems/datasets/ml-100k/u.data', 
                        sep="\t",
                        header=None,
                        usecols=[0, 1, 2],
                        names=["user_id", "movie_id", "rating"]
                        )

# Import data of items
item_df = pd.read_csv('C:/Lecture Notes/Degree/Sem 9/TDA - Algorithm and Analysis/Collaborative-Filtering-Algorithms-in-Recommendation-Systems/datasets/ml-100k/u.item',
                        sep="|",
                        header=None,
                        encoding="latin-1",
                        names=[
                            "movie_id", "title", "release_date", "video_release_date",
                            "IMDb_URL", "unknown", "Action", "Adventure", "Animation",
                            "Children's", "Comedy", "Crime", "Documentary", "Drama", "Fantasy",
                            "Film-Noir", "Horror", "Musical", "Mystery", "Romance", "Sci-Fi",
                            "Thriller", "War", "Western"
                        ])

Select 10,000 sampels out of 100,000 ratings

In [14]:
#select 100k samples
ratings_100k = user_df.head(100000)

print(f"Ratings dataset shape: {ratings_100k.shape}")

Ratings dataset shape: (100000, 3)


### Step 3: Build the user-item matrix

Pivot the long-format ratings table into a wide user-item matrix, where each row is a user, each column is a movie, and each cell contains the user's rating for the movie. This matric is the foundation for both the similarity computation and prediction step later.

In [15]:
#create the user-item matrix
user_item_matrix = ratings_100k.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
)
print(user_item_matrix)

movie_id  1     2     3     4     5     6     7     8     9     10    ...  \
user_id                                                               ...   
1          5.0   3.0   4.0   3.0   3.0   5.0   4.0   1.0   5.0   3.0  ...   
2          4.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   2.0  ...   
3          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
4          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
5          4.0   3.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
...        ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...   
939        NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   5.0   NaN  ...   
940        NaN   NaN   NaN   2.0   NaN   NaN   4.0   5.0   3.0   NaN  ...   
941        5.0   NaN   NaN   NaN   NaN   NaN   4.0   NaN   NaN   NaN  ...   
942        NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
943        NaN   5.0   NaN   NaN   NaN   NaN   NaN   NaN   3.0   NaN  ...   

Since cosine similarity does not operate on missing values, replace NaN with 0

In [16]:
#replace NaN values with 0 
user_item_matrix = user_item_matrix.fillna(0)
print(user_item_matrix.head())

movie_id  1     2     3     4     5     6     7     8     9     10    ...  \
user_id                                                               ...   
1          5.0   3.0   4.0   3.0   3.0   5.0   4.0   1.0   5.0   3.0  ...   
2          4.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   2.0  ...   
3          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
4          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
5          4.0   3.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   

movie_id  1673  1674  1675  1676  1677  1678  1679  1680  1681  1682  
user_id                                                               
1          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
2          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
3          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
4          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
5          0.0   0.0   0.0   0.0  

### Step 4: Compute the user similarity

Calculate the pairwise between every pair of users based on their rating vectors. Users with similar movies rating are close to 1, while users with little to no overlap, or have opposite ratings are closer to 0.

In [17]:
#measure similarity between users (cosine similarity)
user_similarity = cosine_similarity(user_item_matrix)
print(user_similarity)

[[1.         0.16693098 0.04745954 ... 0.14861694 0.17950788 0.39817474]
 [0.16693098 1.         0.11059132 ... 0.16148478 0.17226781 0.10579788]
 [0.04745954 0.11059132 1.         ... 0.10124256 0.13341615 0.02655587]
 ...
 [0.14861694 0.16148478 0.10124256 ... 1.         0.1016418  0.09511958]
 [0.17950788 0.17226781 0.13341615 ... 0.1016418  1.         0.18246466]
 [0.39817474 0.10579788 0.02655587 ... 0.09511958 0.18246466 1.        ]]


### Step 5: Compute the user similarity matrix

Converts the raw similarity to a more readable DataFrame format

In [18]:
# Wrap in a DataFrame for better visualization
user_similarity_df = pd.DataFrame(
    user_similarity,
    index = user_item_matrix.index,
    columns = user_item_matrix.index
).round(4)

print(f"User-User Similarity Matrix shape: {user_similarity_df.shape}\n")
print(user_similarity_df.head())

User-User Similarity Matrix shape: (943, 943)

user_id     1       2       3       4       5       6       7       8    \
user_id                                                                   
1        1.0000  0.1669  0.0475  0.0644  0.3785  0.4302  0.4404  0.3191   
2        0.1669  1.0000  0.1106  0.1781  0.0730  0.2458  0.1073  0.1033   
3        0.0475  0.1106  1.0000  0.3442  0.0212  0.0724  0.0661  0.0831   
4        0.0644  0.1781  0.3442  1.0000  0.0318  0.0680  0.0912  0.1881   
5        0.3785  0.0730  0.0212  0.0318  1.0000  0.2373  0.3736  0.2489   

user_id     9       10   ...     934     935     936     937     938     939  \
user_id                  ...                                                   
1        0.0781  0.3765  ...  0.3695  0.1195  0.2749  0.1897  0.1973  0.1181   
2        0.1610  0.1599  ...  0.1570  0.3079  0.3588  0.4240  0.3199  0.2286   
3        0.0610  0.0652  ...  0.0319  0.0428  0.1638  0.0690  0.1242  0.0263   
4        0.1013  0.0609  ..

### Step 6: Select the target user

Pick a user to generate the recommendations, then split their range into two groups: rated movies and unrated movies. 

In [19]:
# Change user_id to test different users
user = user_item_matrix.index[0] # Example: user_id = 1

# Get the ratings of the target user
rated_movies = user_item_matrix.loc[user]
rated_movies = rated_movies[rated_movies > 0] # Rated movies only

# Unrated movies by the target user - Potential movies for recommendation
unrated_movies = user_item_matrix.loc[user]
unrated_movies = unrated_movies[unrated_movies == 0] # Unrated movies only

print(f"Rated movies by User {user}: {len(rated_movies)}")
print(f"Unrated movies by User {user}: {len(unrated_movies)}")


Rated movies by User 1: 272
Unrated movies by User 1: 1410


### Step 7: Find the top-N similar users

For a given target user and a candidate movie, the function predict_user_ratings finds all other users who rated the movie and compare their similarity to the target user, keeps the top-N most similar, and computes a similarity-weighted average of their ratings as the predicted score. if none of the other users who rated the movie is meaningfully similar, it falls back to the target user's own average rating.

In [20]:
def predict_user_ratings(user_id, item_id, user_item_matrix, user_similarity_df, N=10):
    #get the  users who review this specific movies
    item_ratings = user_item_matrix[item_id]

    raters = item_ratings[item_ratings>0]

    if len(raters) == 0:
        user_ratings = user_item_matrix.loc[user_id]
        return user_ratings[user_ratings>0].mean()
    
    #similarity between target user and each user who rated this movie
    user_similarity_score = user_similarity_df.loc[user_id, raters.index]

    top_n_users = user_similarity_score.sort_values(ascending=False).head(N)

    if top_n_users.sum() == 0:
        user_ratings = user_item_matrix.loc[user_id]
        return user_ratings[user_ratings>0].mean()
    
    numerator = sum(top_n_users[u] * raters[u] for u in top_n_users.index)
    denominator = top_n_users.sum()

    return numerator / denominator

### Step 8: Rank and recommend ratings based on users

Loop the prediction function over every unrated movie for the target user to generate a dictionary of predicted scores. Then, sort and format the Top-K into a readable table with along with the movie titles.

In [21]:
K = 10
N = 10

predicted_scores = {}

for item_id in unrated_movies.index:
    predicted_scores[item_id] = predict_user_ratings(
        user,
        item_id,
        user_item_matrix,
        user_similarity_df,
        N
    )

# Sort predicted scores and return top-K recommended movies
def recommend_movies(predictions, item_df, K):
    # Sort predicted scores in descending order
    top_k = dict(sorted(predictions.items(), key=lambda item: item[1], reverse=True))

    results = []
    for item_id, score in top_k.items():
        title = item_df.loc[item_df['movie_id'] == item_id, 'title'].values
        title = title[0] if len(title) > 0 else "Unknown"
        results.append({
            "movie_id": item_id,
            "title": title,
            "predicted_rating": round(score, 4)
        })
    return pd.DataFrame(results)

recommendations = recommend_movies(predicted_scores, item_df, K)
print(f"Top-{K} Recommended movies for User {user}:\n")
print(recommendations.head(K))


Top-10 Recommended movies for User 1:

   movie_id                                              title  \
0       814                      Great Day in Harlem, A (1994)   
1      1599                      Someone Else's America (1995)   
2      1122                     They Made Me a Criminal (1939)   
3      1189                                 Prefontaine (1997)   
4      1201         Marlene Dietrich: Shadow and Light (1996)    
5      1293                                    Star Kid (1997)   
6      1500                          Santa with Muscles (1996)   
7      1536                               Aiqing wansui (1994)   
8      1653  Entertaining Angels: The Dorothy Day Story (1996)   
9      1467               Saint of Fort Washington, The (1993)   

   predicted_rating  
0               5.0  
1               5.0  
2               5.0  
3               5.0  
4               5.0  
5               5.0  
6               5.0  
7               5.0  
8               5.0  
9             

### Step 9: Evaluate the performance of UBCF

Split a user's ratings to 70% for train (treated as known) and 30% test (hidden from the model). The predicted ratings are compared against the true values.

In [22]:
# Test set = 30%, Train set = 70%
actual = []
predicted = []

start_time = time.time()

for user_id in user_item_matrix.index:
    # Get the ratings of the target user
    user_ratings = user_item_matrix.loc[user_id]
    rated_movies = user_ratings[user_ratings > 0] # Rated movies only

    train_set = rated_movies.sample(frac=0.7) # 70% for train set
    test_set = rated_movies.drop(train_set.index) # Remaining 30% for test set

    for movie_id in test_set.index:
        true_rating = user_item_matrix.loc[user_id, movie_id]
        user_item_matrix.loc[user_id, movie_id] = 0 # Temporarily remove the rating for prediction

        prediction = predict_user_ratings(
            user_id,
            movie_id,
            user_item_matrix,
            user_similarity_df,
            N
        )

        # Restore the original rating
        user_item_matrix.loc[user_id, movie_id] = true_rating

        actual.append(true_rating)
        predicted.append(prediction)

end_time = time.time()

print("Root Mean Squared Error (RMSE):", round(sqrt(mean_squared_error(actual, predicted)), 4))
print("Mean Absolute Error (MAE):", round(mean_absolute_error(actual, predicted), 4))
print("Execution Time:", round(end_time - start_time, 4) * 1000, "ms")

Root Mean Squared Error (RMSE): 1.0119
Mean Absolute Error (MAE): 0.8007
Execution Time: 145419.3 ms
